# Sweep タスクのポリシーを Colab で学習する

ローカルの `make train` と同じ `tools/sweep_train.py` を、同じ lock（`docker/envs/train/uv.lock`）から作った環境で実行します（docs/sweep.md の「Colab での学習」）。トークンは使いません。

**準備（ローカル）**
1. `make colab-data DATASET=sweep_floor` で `data/colab/sweep_floor.tar` を作り、ブラウザで Drive の `MyDrive/leisaac/` に上げる
2. 学習に使う commit を GitHub に push しておく

**実行**
1. ランタイムのタイプを GPU（L4 か A100）にする。T4 は使えない（flash-attn と bf16 に Ampere 以降が要る）
2. 下の「設定」を書き換え、上から順に実行する。Drive のマウントでは許可画面で承認する
3. 最後のセルが `MyDrive/leisaac/<TRAIN_NAME>.tar` を作る。ブラウザでダウンロードし、ローカルで `make import-policy FILE=<tar> TRAIN_NAME=<TRAIN_NAME>` を実行する

In [ ]:
# 設定（ここだけ書き換える）
COMMIT = "origin/main"         # 学習に使う commit（push 済みのもの）。再現のため sha で指定するのがよい
DATASET = "sweep_floor"        # MyDrive/leisaac/<DATASET>.tar（make colab-data の出力）
TRAIN_POLICY = "act"           # act / smolvla / groot
TRAIN_STEPS = 50               # None ならポリシーごとの既定（tools/sweep_train.py）
TRAIN_BATCH = None             # None ならポリシーごとの既定
TRAIN_NAME = f"{DATASET}-{TRAIN_POLICY}"
KEEP_ALL_CHECKPOINTS = False   # False なら最後のチェックポイントだけを Drive に保存する（1 つ 0.6〜数 GB）

DRIVE_DIR = "/content/drive/MyDrive/leisaac"
WORK_DIR = "/content/leisaac"  # Colab のローカルディスク（セッションが終わると消える）
REPO_URL = "https://github.com/Chu-son/leisaac-docker"
UV_VERSION = "0.12.15"         # docker/Dockerfile の UV_IMAGE と合わせる

In [ ]:
# コマンドを実行して出力を表示する補助。失敗したら止める
import json, os, subprocess, sys

def sh(cmd, env=None, cwd=None):
    print(f"+ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"失敗しました（終了コード {p.returncode}）: {cmd}")

def out(cmd):
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout.strip()

## Drive と GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATA_TAR = f"{DRIVE_DIR}/{DATASET}.tar"
RESULT_TAR = f"{DRIVE_DIR}/{TRAIN_NAME}.tar"
assert os.path.isfile(DATA_TAR), f"{DATA_TAR} がありません（make colab-data の出力を Drive に上げる）"
assert not os.path.exists(RESULT_TAR), f"{RESULT_TAR} は既にあります（TRAIN_NAME を変える）"

sh("nvidia-smi")
name, driver, cc = out("nvidia-smi --query-gpu=name,driver_version,compute_cap --format=csv,noheader").split(", ")
GPU = {"name": name, "driver": driver, "compute_cap": cc}
print(GPU)
assert float(cc) >= 8.0, "flash-attn と bf16 には Ampere 以降（L4、A100）が要る。ランタイムのタイプを変える"

## 環境（GitHub から clone し、lock から作る）

In [ ]:
REPO = f"{WORK_DIR}/repo"
VENV = f"{WORK_DIR}/venv"
PY = f"{VENV}/bin/python"
if not os.path.isdir(REPO):
    sh(f"git clone -q {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch -q origin && git -C {REPO} checkout -q --detach {COMMIT}")
REV = out(f"git -C {REPO} rev-parse HEAD")
print("commit", REV)

# lock と pyproject が食い違うと --locked が失敗する。Python 3.11 は uv が取得する
sh(f"{sys.executable} -m pip install -q uv=={UV_VERSION}")
sh(f"{sys.executable} -m uv sync --locked", cwd=f"{REPO}/docker/envs/train",
   env={"UV_PROJECT_ENVIRONMENT": VENV, "VIRTUAL_ENV": VENV, "UV_PYTHON_INSTALL_DIR": f"{WORK_DIR}/python"})

In [ ]:
# torch の CUDA と flash-attn が、この GPU とドライバで動くか（ローカルの make check の 6 番目と同じ確かめ方）
CHECK = """
from importlib.metadata import version
import torch
from flash_attn import flash_attn_func
print("torch", torch.__version__, "CUDA", torch.version.cuda, "lerobot", version("lerobot"), "flash-attn", version("flash-attn"))
assert torch.cuda.is_available(), "torch から GPU が見えない (ドライバが cu128 の torch に対応していない可能性)"
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
x = torch.randn(256, 256, device="cuda")
assert torch.isfinite(x @ x).all()
q = torch.randn(1, 128, 8, 64, device="cuda", dtype=torch.bfloat16)
assert flash_attn_func(q, q, q).shape == q.shape
torch.cuda.synchronize()
print("ENV OK")
"""
with open(f"{WORK_DIR}/env_check.py", "w") as f:
    f.write(CHECK)
sh(f"{PY} {WORK_DIR}/env_check.py")

## 学習

In [ ]:
DATASET_ROOT = f"{WORK_DIR}/data/lerobot/{DATASET}"
if not os.path.isdir(DATASET_ROOT):
    os.makedirs(f"{WORK_DIR}/data/lerobot", exist_ok=True)
    sh(f"tar -C {WORK_DIR}/data/lerobot -xf {DATA_TAR}")
assert os.path.isfile(f"{DATASET_ROOT}/meta/info.json"), f"{DATA_TAR} の中身が {DATASET}/ ではない"

# 出力先が既にあれば sweep_train.py が止める（上書きも再開もしない）。やり直すなら TRAIN_NAME を変える
OUTPUT_DIR = f"{WORK_DIR}/train/{TRAIN_NAME}"
args = f"--policy={TRAIN_POLICY} --dataset_root={DATASET_ROOT} --output_dir={OUTPUT_DIR}"
if TRAIN_STEPS:
    args += f" --steps={TRAIN_STEPS}"
if TRAIN_BATCH:
    args += f" --batch_size={TRAIN_BATCH}"
# モデルの重み（Hub と torchvision）は Colab のローカルディスクに置く
sh(f"{PY} {REPO}/tools/sweep_train.py {args}",
   env={"PYTHONPATH": f"{REPO}/tools", "HF_HOME": f"{WORK_DIR}/cache", "TORCH_HOME": f"{WORK_DIR}/cache/torch",
        "VIRTUAL_ENV": VENV, "PATH": f"{VENV}/bin:{os.environ['PATH']}"})

## Drive に保存

In [ ]:
# 中身は sweep_train.py の output_dir（make import-policy は ./sweep_train.json があることを確かめる）。
# colab.json に、使った commit と GPU を残す
with open(f"{OUTPUT_DIR}/colab.json", "w") as f:
    json.dump({"commit": REV, "gpu": GPU, "train_policy": TRAIN_POLICY, "train_steps": TRAIN_STEPS,
               "train_batch": TRAIN_BATCH, "dataset": DATASET}, f, indent=2)
if KEEP_ALL_CHECKPOINTS:
    members = "."
else:
    last = os.readlink(f"{OUTPUT_DIR}/checkpoints/last")
    members = f"./sweep_train.json ./colab.json ./checkpoints/last ./checkpoints/{last}"
# 書き込みの途中で切れたものを残さないよう、別名で書いてから名前を変える
sh(f"tar -C {OUTPUT_DIR} -cf {RESULT_TAR}.part {members} && mv {RESULT_TAR}.part {RESULT_TAR}")
sh(f"ls -lh {RESULT_TAR}")
print(f"ダウンロードしたら: make import-policy FILE=<ダウンロードした {TRAIN_NAME}.tar> TRAIN_NAME={TRAIN_NAME}")